# Community Notes 陣営反応分析 — 全データ実行

## 手順
1. このノートブックを Colab で開く
2. 上から順にセルを実行

データは共有ドライブ「基礎プロジェクト/data/」から自動でコピーします。

## 0. セットアップ

In [ ]:
# Google Drive をマウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess

# ====================================================
# ★ Drive のデータフォルダのパスをここで設定 ★
# ====================================================
DRIVE_DATA = '/content/drive/Shareddrives/基礎プロジェクト/data'

# パスが正しいか確認
if os.path.exists(DRIVE_DATA):
    print('OK: フォルダが見つかりました')
    for d in sorted(os.listdir(DRIVE_DATA)):
        print(f'  {d}')
else:
    print(f'ERROR: {DRIVE_DATA} が見つかりません')
    print()
    print('--- 共有ドライブ一覧 ---')
    sd = '/content/drive/Shareddrives'
    if os.path.exists(sd):
        for d in os.listdir(sd):
            print(f'  {d}')
    else:
        print('  共有ドライブなし')
    print()
    print('--- マイドライブ ---')
    for d in sorted(os.listdir('/content/drive/MyDrive'))[:20]:
        print(f'  {d}')

In [ ]:
# リポジトリをクローン
!git clone https://github.com/hirototoda/toriumi_x3.git /content/toriumi_x3 2>/dev/null || echo 'already cloned'
os.chdir('/content/toriumi_x3')
!git pull
!pwd

In [ ]:
# 依存パッケージ
!pip install -q statsmodels

In [ ]:
# Drive の zip を data/raw/ に解凍
raw_dir = '/content/toriumi_x3/data/raw'
os.makedirs(raw_dir, exist_ok=True)

# 解凍対象のサブフォルダとファイルパターン
targets = [
    ('RatingData', 'ratings-'),
    ('Notes data', 'notes-'),
    ('Note status history data', 'noteStatusHistory-'),
]

for subfolder, prefix in targets:
    folder = os.path.join(DRIVE_DATA, subfolder)
    if not os.path.exists(folder):
        print(f'WARNING: {folder} が見つかりません、スキップ')
        continue
    for f in sorted(os.listdir(folder)):
        if f.startswith(prefix) and f.endswith('.zip'):
            zip_path = os.path.join(folder, f)
            print(f'Unzipping {f} ...')
            subprocess.run(['unzip', '-o', '-q', zip_path, '-d', raw_dir], check=True)

print()
print('=== data/raw/ ===')
for f in sorted(os.listdir(raw_dir)):
    if f.endswith('.tsv'):
        size = os.path.getsize(os.path.join(raw_dir, f)) / (1024**3)
        print(f'  {f}  ({size:.1f} GB)')

## 1. 全トピックでパイプライン実行

In [ ]:
!python scripts/run_pipeline.py

## 2. トピック別比較

In [ ]:
!python scripts/run_by_topic.py

## 3. 結果の確認

In [ ]:
import pandas as pd

# トピック別比較表
df = pd.read_csv('data/processed/topic_comparison.csv')
display(df)

In [ ]:
# ターゲットノート
targets = pd.read_csv('data/processed/target_notes.csv')
print(f'ターゲットノート数: {len(targets)}')
display(targets.head(20))

In [ ]:
# バースト一覧
bursts = pd.read_csv('data/processed/bursts.csv')
print(f'バースト数: {len(bursts)}')
print(bursts['burst_type'].value_counts())

## 4. 結果を Drive に保存

In [ ]:
save_dir = os.path.join(DRIVE_DATA, '..', 'results')
os.makedirs(save_dir, exist_ok=True)
for f in os.listdir('data/processed'):
    src = os.path.join('data/processed', f)
    if f.endswith('.csv') or f.endswith('.png'):
        subprocess.run(['cp', src, save_dir])
print(f'Done! Results saved to: {os.path.abspath(save_dir)}')